# Papeer: Qdrant + OpenRouter Embedding Diagnostic

This notebook tests the pipeline **outside Streamlit**:

1. Find the project root, `.env`, and PDF.
2. Load and split `documents/Openclaw_Research_Report.pdf`.
3. Test OpenRouter embeddings.
4. Test Qdrant connectivity.
5. Create an isolated diagnostic collection, insert a few chunks, and run similarity search.
6. Optionally delete only the diagnostic collection.

## Your project layout

Your project root is:

`E:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\15_rag_papeer_project`

Your `.venv`, `.env`, and `documents` folder are inside that root. Your notebook is inside the `notebooks` subfolder. The first code cell automatically detects this layout whether the notebook starts from the project root or `notebooks`.

**Security:** keep your API keys in `.env`. Do not paste the keys into notebook cells or share screenshots that reveal them.

## 0. Install the required packages

Run this cell using the kernel for your project's `.venv`. In VS Code, choose **Select Kernel** and select the Python interpreter inside your project's `.venv`. You do not need to create another virtual environment.

In [1]:
# %pip install -U python-dotenv langchain-community langchain-core langchain-text-splitters langchain-openai langchain-qdrant qdrant-client pymupdf

## 1. Detect the project root and load `.env`

This cell searches the current working directory and its parent folders for a folder containing both `.env` and `documents`. It then loads that root's `.env`.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

START_DIR = Path.cwd().resolve()

def find_project_root(start: Path) -> Path:
    # Search current folder and ancestors first.
    for candidate in (start, *start.parents):
        if (candidate / ".env").is_file() and (candidate / "documents").is_dir():
            return candidate

    # If the notebook's working directory is not under the project root,
    # try the expected project path supplied for this project.
    expected = Path(
        r"E:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\15_rag_papeer_project"
    )
    if (expected / ".env").is_file() and (expected / "documents").is_dir():
        return expected

    raise FileNotFoundError(
        "Could not find the project root. Expected a folder containing both "
        "'.env' and 'documents'. Current working directory: "
        f"{start}\nSet PROJECT_ROOT manually in this cell."
    )

PROJECT_ROOT = find_project_root(START_DIR)
ENV_FILE = PROJECT_ROOT / ".env"
PDF_PATH = PROJECT_ROOT / "documents" / "Openclaw_Research_Report.pdf"

# Do not print secret values.
load_dotenv(ENV_FILE, override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
OPENROUTER_BASE_URL = os.getenv(
    "OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"
).strip()
EMBEDDING_MODEL = os.getenv(
    "OPENROUTER_EMBEDDING_MODEL",
    "nvidia/nemotron-3.5-embed-1b:free"
).strip()
QDRANT_URL = os.getenv("QDRANT_URL", "").strip()
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "").strip()

print("Notebook started from:", START_DIR)
print("Detected project root:", PROJECT_ROOT)
print(".env exists:", ENV_FILE.is_file())
print("PDF exists:", PDF_PATH.is_file())
print("OpenRouter API key configured:", bool(OPENROUTER_API_KEY))
print("OpenRouter base URL:", OPENROUTER_BASE_URL)
print("Embedding model:", EMBEDDING_MODEL)
print("Qdrant URL configured:", bool(QDRANT_URL))
print("Qdrant API key configured:", bool(QDRANT_API_KEY))

assert ENV_FILE.is_file(), f".env not found: {ENV_FILE}"
assert PDF_PATH.is_file(), f"PDF not found: {PDF_PATH}"

Notebook started from: E:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\15_rag_papeer_project\notebooks
Detected project root: E:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\15_rag_papeer_project
.env exists: True
PDF exists: True
OpenRouter API key configured: True
OpenRouter base URL: https://openrouter.ai/api/v1
Embedding model: nvidia/nemotron-3.5-embed-1b:free
Qdrant URL configured: True
Qdrant API key configured: True


## 2. Check required credentials

This cell stops early if a required setting is missing. It never displays the secret values.

In [3]:
missing_settings = []
if not OPENROUTER_API_KEY:
    missing_settings.append("OPENROUTER_API_KEY")
if not QDRANT_URL:
    missing_settings.append("QDRANT_URL")
if not QDRANT_API_KEY:
    missing_settings.append("QDRANT_API_KEY")

if missing_settings:
    raise RuntimeError(
        "Missing values in the project .env: " + ", ".join(missing_settings)
    )

print("Required credentials are present.")

Required credentials are present.


## 3. Load and inspect the PDF

This step is local. It does not call OpenRouter or Qdrant.

In [4]:
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader(str(PDF_PATH))
raw_docs = loader.load()

print("PDF path:", PDF_PATH)
print("Pages loaded:", len(raw_docs))
print("First page metadata:", raw_docs[0].metadata if raw_docs else None)
print("\nFirst page preview:\n")
print(raw_docs[0].page_content[:800] if raw_docs else "No pages extracted.")

assert raw_docs, "PDF loader returned no pages."
assert any(page.page_content.strip() for page in raw_docs), (
    "PDF pages were found, but no text was extracted."
)

PDF path: E:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\15_rag_papeer_project\documents\Openclaw_Research_Report.pdf
Pages loaded: 34
First page metadata: {'producer': 'Skia/PDF m127', 'creator': 'Chromium', 'creationdate': '2026-01-31T03:22:31+00:00', 'source': 'E:\\Projects\\Campusx-Advance-Rag\\Code\\Advanced_Rag_Codes\\15_rag_papeer_project\\documents\\Openclaw_Research_Report.pdf', 'file_path': 'E:\\Projects\\Campusx-Advance-Rag\\Code\\Advanced_Rag_Codes\\15_rag_papeer_project\\documents\\Openclaw_Research_Report.pdf', 'total_pages': 34, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-01-31T03:22:31+00:00', 'trapped': '', 'modDate': "D:20260131032231+00'00'", 'creationDate': "D:20260131032231+00'00'", 'page': 0}

First page preview:

Report Date: January 31, 2026
Prepared for: Technical Review and Strategic Assessment
Report Version: 1.0
Moltbot (formerly Clawdbot) represents one of the most viral open-source AI projects in

## 4. Split the PDF into chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
docs = splitter.split_documents(raw_docs)

print("Chunks created:", len(docs))
print("First chunk metadata:", docs[0].metadata if docs else None)
print("\nFirst chunk preview:\n")
print(docs[0].page_content[:500] if docs else "No chunks created.")

assert docs, "No chunks were created."

Chunks created: 92
First chunk metadata: {'producer': 'Skia/PDF m127', 'creator': 'Chromium', 'creationdate': '2026-01-31T03:22:31+00:00', 'source': 'E:\\Projects\\Campusx-Advance-Rag\\Code\\Advanced_Rag_Codes\\15_rag_papeer_project\\documents\\Openclaw_Research_Report.pdf', 'file_path': 'E:\\Projects\\Campusx-Advance-Rag\\Code\\Advanced_Rag_Codes\\15_rag_papeer_project\\documents\\Openclaw_Research_Report.pdf', 'total_pages': 34, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-01-31T03:22:31+00:00', 'trapped': '', 'modDate': "D:20260131032231+00'00'", 'creationDate': "D:20260131032231+00'00'", 'page': 0}

First chunk preview:

Report Date: January 31, 2026
Prepared for: Technical Review and Strategic Assessment
Report Version: 1.0
Moltbot (formerly Clawdbot) represents one of the most viral open-source AI projects in
GitHub history, achieving over 103,000 stars within three months of launch[32]. This self-
hosted, agentic AI assistant ha

## 5. Test OpenRouter embeddings without Qdrant

The chat model `google/gemma-4-31b-it:free` is not the embedding model. This notebook uses `OPENROUTER_EMBEDDING_MODEL` if you have set it in `.env`; otherwise it uses `nvidia/nemotron-3.5-embed-1b:free`.

If the provider/model rejects the request, check the full error before moving on. The notebook does not assume that every model marked `:free` supports embeddings.

In [6]:
from math import isfinite
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model=EMBEDDING_MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    check_embedding_ctx_length=False,
    encoding_format="float",
)

query_vector = embedding_model.embed_query(
    "Test embedding request for the Papeer Qdrant diagnostic."
)
print("Query embedding succeeded.")
print("Model:", EMBEDDING_MODEL)
print("Vector dimension:", len(query_vector))
print("All vector values are finite:", all(isfinite(value) for value in query_vector))

chunk_vector = embedding_model.embed_documents([docs[0].page_content])[0]
print("Embedding a PDF chunk succeeded.")
print("PDF chunk vector dimension:", len(chunk_vector))

assert len(query_vector) == len(chunk_vector), (
    "Query and document embedding dimensions do not match."
)
EMBEDDING_DIM = len(query_vector)
print("Embedding dimension to use for the Qdrant test collection:", EMBEDDING_DIM)

e:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\15_rag_papeer_project\.venv\Lib\site-packages\langchain_openai\embeddings\base.py:355: UserWarning: WARNING! encoding_format is not default parameter.
                    encoding_format was transferred to model_kwargs.
                    Please confirm that encoding_format is what you intended.
  warnings.warn(


Query embedding succeeded.
Model: nvidia/nemotron-3.5-embed-1b:free
Vector dimension: 2048
All vector values are finite: True
Embedding a PDF chunk succeeded.
PDF chunk vector dimension: 2048
Embedding dimension to use for the Qdrant test collection: 2048


## 6. Test Qdrant connectivity only

This only requests the list of collections. It does not create, modify, or delete anything.

In [7]:
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=30,
)

response = qdrant_client.get_collections()
print("Qdrant connection succeeded.")
print("Collections returned:", len(response.collections))
for collection in response.collections:
    print(" -", collection.name)

Qdrant connection succeeded.
Collections returned: 0


e:\Projects\Campusx-Advance-Rag\Code\Advanced_Rag_Codes\15_rag_papeer_project\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:280: UserWarning: Qdrant client version 1.17.1 is incompatible with server version 1.19.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


## 7. Create a separate diagnostic collection

This collection name is unique and starts with `papeer_diag_`. It will not overwrite the app's session collections.

In [8]:
import uuid
from qdrant_client.models import Distance, VectorParams

TEST_COLLECTION = f"papeer_diag_{uuid.uuid4().hex[:12]}"
print("Diagnostic collection name:", TEST_COLLECTION)

qdrant_client.create_collection(
    collection_name=TEST_COLLECTION,
    vectors_config=VectorParams(
        size=EMBEDDING_DIM,
        distance=Distance.COSINE,
    ),
)
print("Collection created successfully.")
print("Collection vector config:", qdrant_client.get_collection(TEST_COLLECTION).config.params.vectors)

Diagnostic collection name: papeer_diag_1673c6689b8a
Collection created successfully.
Collection vector config: size=2048 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None datatype=None multivector_config=None


## 8. Insert a small sample and run similarity search

Only the first five chunks are inserted to keep this diagnostic small and limit embedding usage.

In [9]:
from langchain_qdrant import QdrantVectorStore

TEST_CHUNK_COUNT = min(5, len(docs))
test_docs = docs[:TEST_CHUNK_COUNT]

vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name=TEST_COLLECTION,
    embedding=embedding_model,
)

inserted_ids = vector_store.add_documents(test_docs)
print("Chunks submitted:", TEST_CHUNK_COUNT)
print("IDs returned:", len(inserted_ids))

query = "What is the main subject of this research report?"
results = vector_store.similarity_search(query, k=min(3, TEST_CHUNK_COUNT))

print("\nResults returned:", len(results))
for i, result in enumerate(results, start=1):
    print(f"\n--- Result {i} ---")
    print("Metadata:", result.metadata)
    print(result.page_content[:700])

Chunks submitted: 5
IDs returned: 5

Results returned: 3

--- Result 1 ---
Metadata: {'producer': 'Skia/PDF m127', 'creator': 'Chromium', 'creationdate': '2026-01-31T03:22:31+00:00', 'source': 'E:\\Projects\\Campusx-Advance-Rag\\Code\\Advanced_Rag_Codes\\15_rag_papeer_project\\documents\\Openclaw_Research_Report.pdf', 'file_path': 'E:\\Projects\\Campusx-Advance-Rag\\Code\\Advanced_Rag_Codes\\15_rag_papeer_project\\documents\\Openclaw_Research_Report.pdf', 'total_pages': 34, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-01-31T03:22:31+00:00', 'trapped': '', 'modDate': "D:20260131032231+00'00'", 'creationDate': "D:20260131032231+00'00'", 'page': 0, '_id': '658cdc94-c9d6-4c1a-a879-090b9257507e', '_collection_name': 'papeer_diag_1673c6689b8a'}
Report Date: January 31, 2026
Prepared for: Technical Review and Strategic Assessment
Report Version: 1.0
Moltbot (formerly Clawdbot) represents one of the most viral open-source AI projects in
GitHub

## 9. Verify that Qdrant stored points

In [10]:
collection_info = qdrant_client.get_collection(TEST_COLLECTION)
print("Diagnostic collection:", TEST_COLLECTION)
print("Stored point count:", collection_info.points_count)
print("Expected at least:", TEST_CHUNK_COUNT)

assert (collection_info.points_count or 0) >= TEST_CHUNK_COUNT, (
    "The stored point count is lower than expected."
)
print("PASS: PDF loading, embedding, Qdrant insertion, and similarity search completed.")

Diagnostic collection: papeer_diag_1673c6689b8a
Stored point count: 5
Expected at least: 5
PASS: PDF loading, embedding, Qdrant insertion, and similarity search completed.


## 10. Optional cleanup

Run this cell only after you have finished inspecting the diagnostic collection. It deletes only the temporary collection created in cell 7. It does not delete any `papeer_<session-id>` app collections.

In [11]:
assert TEST_COLLECTION.startswith("papeer_diag_"), (
    "Safety check failed; refusing to delete a collection with an unexpected name."
)
qdrant_client.delete_collection(TEST_COLLECTION)
print("Deleted temporary diagnostic collection:", TEST_COLLECTION)

Deleted temporary diagnostic collection: papeer_diag_1673c6689b8a


## How to interpret a failure

- **Step 1:** path or `.env` detection problem.
- **Step 3:** local PDF loading/extraction problem.
- **Step 5:** OpenRouter key, model availability, embedding endpoint, or request-format problem.
- **Step 6:** Qdrant URL, API key, cluster/network, TLS, or connectivity problem.
- **Step 7:** Qdrant permissions or collection-creation problem.
- **Step 8:** embedding request, vector dimension, insertion, or search problem.

Run from top to bottom and focus on the **first failing cell**. Keep its full traceback, but redact API keys and other secrets before sharing it.